In [ ]:
from flowrec.data.yambda import YambdaDataConfig, YambdaDataModule

In [ ]:
cfg = YambdaDataConfig(
    use_hf=True,
    dataset_type="flat",
    dataset_size="50m",
    event_file="listens",
    min_played_ratio_pct=50,
    split_strategy="leave_one_out",
    min_user_interactions=5,
    min_item_interactions=5,
    max_history_len=128,
    batch_size=256,
    keep_raw_interactions=True,
    global_eval_target_mode="last_per_user",
    train_all_prefixes=False
)

dm = YambdaDataModule(cfg)
dm.setup()

print(dm.summary())
batch = next(iter(dm.train_dataloader()))
print(batch["history_item_ids"].shape)

In [ ]:
from flowrec.item_representation.yambda import YambdaAudioEmbeddingConfig, YambdaAudioEmbeddingBuilder

store = YambdaAudioEmbeddingBuilder(
    YambdaAudioEmbeddingConfig(
        embeddings_path="../data/raw/yambda/embeddings.parquet",
        embedding_column="normalized_embed",
    )
).build()

store.save("../artifacts/item_embeddings/yambda_audio_embeddings.pt")

In [ ]:
from flowrec.item_representation.content_embeddings import ItemEmbeddingStore
store = ItemEmbeddingStore.load("../artifacts/item_embeddings/yambda_audio_embeddings.pt")

In [ ]:
from flowrec.item_representation.semantic_id import (
    RQVAEConfig,
    RQVAETrainConfig,
    RQVAEItemTokenizerConfig,
    RQVAEItemTokenizer,
)


# Keep only the items that remain in the final dataset after preprocessing, then
# train the RQ-VAE tokenizer on exactly those content embeddings.
store_in_data = store.subset(dm.item_ids_in_data())
print(f"{store_in_data.num_items} / {store.num_items} items remain in the final dataset")

tokenizer = RQVAEItemTokenizer(
    RQVAEItemTokenizerConfig(
        rqvae=RQVAEConfig(
            input_dim=store_in_data.embeddings.shape[1],
            latent_dim=32,
            num_codebooks=3,
            codebook_size=256,
        ),
        train=RQVAETrainConfig(epochs=50, device="cpu"),
        normalize="l2",
        collision_policy="append",
    )
)

tokenizer.fit(store_in_data.embeddings, store_in_data.item_ids)
tokenizer.save("../artifacts/tokenizers/yambda_rqvae")

semantic_id_mapping = tokenizer.semantic_id_store.to_plain_dict()

In [ ]:
# Attach the trained semantic IDs to the datamodule that was already built and
# preprocessed above, instead of constructing a second instance from a saved
# semantic_ids.json path.
dm.add_semantic_ids(semantic_ids=semantic_id_mapping)

batch = next(iter(dm.train_dataloader()))
print(batch["history_token_ids"].shape)
print(batch["target_token_ids"].shape)